# Phase 2 — Standard QAT (INT8 & INT4) — Google Colab

**Goal:** measure what one full pass of QAT with fake-quantization-from-step-1 achieves. This is the standard baseline that Scheduled QAT (notebook 04) has to beat to justify the schedule.

**Inputs (from your Drive):**
- `/content/drive/MyDrive/sqat-baseline/results/baseline/fp32_logits.pt`

**Outputs (saved at the end to `/content/drive/MyDrive/sqat-standard-qat/`):**
- `models/checkpoints/standard_qat_int{4,8}.pt` — final trained models, renamed for portability
- `results/standard_qat_int{4,8}/training_results.json` — perplexity, KLD, training stats
- `results/standard_qat_inference.json` — 5-prompt sample completions
- `results/logs/qat_int{4,8}/per_step_loss.jsonl` — **micro view** (loss every optimizer step)
- `results/logs/qat_int{4,8}/training_steps.jsonl` — **macro view** (loss + val PPL at eval intervals)
- `data_samples/{train,val,test}_sample.pt` — small samples of the actual tokenized data used

**Estimated time on Colab T4:** ~3.5 h total (≈1.7 h per bit-width)."

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

WORKING_DIR  = "/content"
REPO_DIR     = f"{WORKING_DIR}/scheduled-qat-for-slm"
GITHUB_URL   = "https://github.com/JpCurada/scheduled-qat-for-slm.git"
BASELINE_DIR = "/content/drive/MyDrive/sqat-baseline/results/baseline"
DRIVE_ROOT   = "/content/drive/MyDrive/sqat-standard-qat"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

FP32_LOGITS = Path(BASELINE_DIR) / "fp32_logits.pt"
assert FP32_LOGITS.exists(), (
    f"FP32 logits not found at {FP32_LOGITS}. "
    "Run notebook 01 first and ensure outputs were saved to "
    "/content/drive/MyDrive/sqat-baseline/."
)
print(f"FP32 logits: {FP32_LOGITS}  ({FP32_LOGITS.stat().st_size / 1e9:.2f} GB)")

In [ ]:
!pip install -q -e ".[viz]"

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")
if torch.cuda.is_available():
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Kaggle config overrides

The published configs assume `seq_length=2048`, `epochs=3`, `batch_size=8`. That doesn't fit Kaggle's 12 h session limit on a T4 — at the published settings, INT4 alone takes ~15 h. We patch:

| Field | Original | Kaggle |
|---|---|---|
| `data.seq_length` | 2048 | 512 |
| `training.epochs` | 3 | 1 |
| `training.batch_size` | 8 | 4 |
| `training.gradient_accumulation_steps` | 4 | 8 (effective batch unchanged at 32) |
| `training.warmup_steps` | 500 | 100 |
| `logging.save_every_steps` | 5000 | 999999 (only final checkpoint) |
| `logging.eval_every_steps` | 2500 | 500 |

Effective batch size and learning rate are preserved so results are still comparable to the full-spec runs.

In [ ]:
import yaml

# Reuse the FP32 model cache from sqat-baseline if it's there.
DRIVE_MODEL_CACHE = "/content/drive/MyDrive/sqat-baseline/models/base"
LOCAL_MODEL_CACHE = f"{REPO_DIR}/models/base"
MODEL_CACHE       = DRIVE_MODEL_CACHE if Path(DRIVE_MODEL_CACHE).exists() else LOCAL_MODEL_CACHE

COLAB_CFG = Path(REPO_DIR) / "configs_colab" / "standard_qat"
COLAB_CFG.mkdir(parents=True, exist_ok=True)

def patch_qat_config(src: str, dst: Path) -> Path:
    with open(src) as f:
        cfg = yaml.safe_load(f)
    cfg["model"]["cache_dir"] = MODEL_CACHE
    cfg["data"]["seq_length"] = 512
    cfg["training"]["epochs"] = 1
    cfg["training"]["batch_size"] = 4
    cfg["training"]["gradient_accumulation_steps"] = 8
    cfg["training"]["warmup_steps"] = 100
    cfg["logging"]["save_every_steps"] = 999999
    cfg["logging"]["eval_every_steps"] = 500
    cfg["logging"]["log_dir"] = f"{REPO_DIR}/results/logs/{dst.stem}/"
    cfg["export"]["output_dir"] = f"{REPO_DIR}/models/gguf/"
    with dst.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    return dst

qat8_cfg = patch_qat_config("configs/standard_qat/qat_int8.yaml", COLAB_CFG / "qat_int8.yaml")
qat4_cfg = patch_qat_config("configs/standard_qat/qat_int4.yaml", COLAB_CFG / "qat_int4.yaml")
print("INT8 config:", qat8_cfg)
print("INT4 config:", qat4_cfg)
print(f"Model cache: {MODEL_CACHE}")

## 3. Standard QAT — INT8

INT8 is the easier target — fake-quant noise is small enough that even one epoch should recover most FP32 quality. Look for KL divergence noticeably below PTQ INT8.

In [ ]:
import gc
from src.training.trainer import run_experiment

results_int8 = run_experiment(
    config_path=str(qat8_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nStandard QAT INT8 results:")
for k, v in results_int8.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 4. Standard QAT — INT4

INT4 is where QAT earns its keep. The fake-quant noise is large from step 1; expect higher loss and noticeable KL divergence vs the FP32 baseline. This is the headline number that Scheduled QAT will try to beat.

In [ ]:
results_int4 = run_experiment(
    config_path=str(qat4_cfg),
    device_str="cuda",
    baseline_logits=str(FP32_LOGITS),
    run_benchmarks=False,
)

print("\nStandard QAT INT4 results:")
for k, v in results_int4.items():
    print(f"  {k:25s}  {v}")

gc.collect(); torch.cuda.empty_cache()

## 5. Training loss curves

Read JSONL step-logs from both runs and overlay the loss / val-perplexity curves. Helps identify if the training plateaus, diverges, or still has headroom.

In [ ]:
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_jsonl(path):
    if not Path(path).exists():
        return []
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

def plot_loss_curves(per_step_logs, eval_logs, titles):
    """Two-panel comparison: micro view (per-step loss) + macro view (val PPL)."""
    fig = make_subplots(rows=1, cols=len(per_step_logs),
                        subplot_titles=titles,
                        specs=[[{"secondary_y": True}] * len(per_step_logs)])
    for col, (mlog, elog) in enumerate(zip(per_step_logs, eval_logs), 1):
        if mlog:
            fig.add_trace(go.Scatter(
                x=[e["step"] for e in mlog], y=[e["loss"] for e in mlog],
                name="train loss (per step)",
                line=dict(color="#4C72B0", width=1), opacity=0.6,
                legendgroup=str(col), showlegend=(col == 1),
            ), row=1, col=col, secondary_y=False)
        if elog:
            fig.add_trace(go.Scatter(
                x=[e["step"] for e in elog], y=[e.get("val_ppl") for e in elog],
                name="val ppl (eval interval)",
                line=dict(color="#DD8452", dash="dash", width=2),
                mode="lines+markers", legendgroup=str(col), showlegend=(col == 1),
            ), row=1, col=col, secondary_y=True)
        fig.update_xaxes(title_text="step", row=1, col=col)
        fig.update_yaxes(title_text="loss", row=1, col=col, secondary_y=False)
        fig.update_yaxes(title_text="val perplexity", row=1, col=col, secondary_y=True)
    fig.update_layout(title="Standard QAT — training curves (micro + macro)",
                      height=420, hovermode="x unified",
                      margin=dict(t=80, b=40, l=60, r=60))
    fig.show()

mlog8 = read_jsonl(f"{REPO_DIR}/results/logs/qat_int8/per_step_loss.jsonl")
mlog4 = read_jsonl(f"{REPO_DIR}/results/logs/qat_int4/per_step_loss.jsonl")
elog8 = read_jsonl(f"{REPO_DIR}/results/logs/qat_int8/training_steps.jsonl")
elog4 = read_jsonl(f"{REPO_DIR}/results/logs/qat_int4/training_steps.jsonl")
plot_loss_curves([mlog8, mlog4], [elog8, elog4], ["INT8", "INT4"])

## 6. Comparison table

In [ ]:
import pandas as pd

with open(Path(BASELINE_DIR) / "baseline_results.json") as f:
    fp32 = json.load(f)

rows = [
    {"method": "FP32",         "bits": 32, **{k: fp32.get(k)         for k in ("perplexity",)}, "kl_divergence": 0.0},
    {"method": "Standard QAT", "bits": 8,  "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence")},
    {"method": "Standard QAT", "bits": 4,  "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence")},
]
df = pd.DataFrame(rows)
df["ppl_delta_pct"] = ((df["perplexity"] / df["perplexity"].iloc[0]) - 1.0) * 100
df.round(4)

In [ ]:
summary_path = Path(REPO_DIR) / "results" / "standard_qat_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
with summary_path.open("w") as f:
    json.dump({"int8": results_int8, "int4": results_int4, "fp32": fp32},
              f, indent=2, default=str)
print(f"Wrote {summary_path}")
!ls -lh {REPO_DIR}/models/checkpoints/

## 7. Plotly comparison — quality vs FP32

Identical chart to notebook 02 — same axes, same colour scheme, so visual comparison across notebooks is direct.

In [ ]:
def plot_method_comparison(rows, title, fp32_ppl=None):
    labels = [r["label"] for r in rows]
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=("Perplexity (lower=better)",
                                        "KL Divergence vs FP32 (lower=better)",
                                        "Model size (GB)"))
    fig.add_trace(go.Bar(x=labels, y=[r["perplexity"]    for r in rows],
                         marker_color="#4C72B0",
                         text=[f"{r['perplexity']:.3f}" if r["perplexity"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 1)
    if fp32_ppl is not None:
        fig.add_hline(y=fp32_ppl, line_dash="dash", line_color="black",
                      annotation_text=f"FP32 = {fp32_ppl:.3f}", row=1, col=1)
    fig.add_trace(go.Bar(x=labels, y=[r["kl_divergence"] for r in rows],
                         marker_color="#DD8452",
                         text=[f"{r['kl_divergence']:.4f}" if r["kl_divergence"] is not None else "—" for r in rows],
                         textposition="outside"), 1, 2)
    fig.add_trace(go.Bar(x=labels, y=[r["size_gb"]       for r in rows],
                         marker_color="#55A868",
                         text=[f"{r['size_gb']:.2f}" for r in rows],
                         textposition="outside"), 1, 3)
    fig.update_layout(title=title, showlegend=False, height=420,
                      margin=dict(t=80, b=40, l=40, r=20))
    fig.show()

rows = [
    {"label": "FP32 baseline",     "perplexity": fp32.get("perplexity"),         "kl_divergence": 0.0,                                 "size_gb": 6.5},
    {"label": "Standard QAT INT8", "perplexity": results_int8.get("perplexity"), "kl_divergence": results_int8.get("kl_divergence"),   "size_gb": 1.7},
    {"label": "Standard QAT INT4", "perplexity": results_int4.get("perplexity"), "kl_divergence": results_int4.get("kl_divergence"),   "size_gb": 0.85},
]
plot_method_comparison(rows, "Standard QAT — quantization quality vs FP32",
                       fp32_ppl=fp32.get("perplexity"))

## 8. Sample inference

Same five prompts and helper as notebook 02 — direct apples-to-apples comparison across notebooks.

For QAT methods we reload the trained model from the saved checkpoint:
1. `build_model_for_training(cfg, device)` — re-creates the architecture with fake-quant nodes.
2. `load_checkpoint(model, final.pt)` — loads the trained shadow weights.
3. Run greedy generation.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from src.models.model_wrapper import build_model_for_training
from src.utils.config_loader import load_config
from src.quantization.standard_qat import load_checkpoint as load_qat_ckpt

SAMPLE_PROMPTS = [
    "The capital of France is",
    "Python is a programming language used for",
    "Once upon a time, in a small village,",
    "The chemical symbol for gold is",
    "To improve your sleep quality, you should",
]
MAX_NEW_TOKENS = 30
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def generate_with_model(model, prompts=SAMPLE_PROMPTS, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    out = []
    for p in prompts:
        ids = tokenizer(p, return_tensors="pt").input_ids.to(device)
        with torch.no_grad():
            gen = model.generate(
                ids, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        out.append(tokenizer.decode(gen[0][ids.shape[1]:], skip_special_tokens=True).strip())
    return out

def free():
    gc.collect(); torch.cuda.empty_cache()

# 1. FP32 baseline
print("Generating FP32 (FP16 weights for memory) ...")
fp32_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B", cache_dir=MODEL_CACHE, torch_dtype=torch.float16,
).to(device)
fp32_outs = generate_with_model(fp32_model)
del fp32_model; free()

def load_qat_for_inference(cfg_path, ckpt_path):
    cfg = load_config(str(cfg_path)); cfg.model.cache_dir = MODEL_CACHE
    wrap = build_model_for_training(cfg, device)
    if Path(ckpt_path).exists():
        load_qat_ckpt(wrap.model, ckpt_path)
    else:
        print(f"  WARNING — checkpoint not found at {ckpt_path}; using freshly-quantized weights")
    return wrap.model

# 2. Standard QAT INT8
print("Generating Standard QAT INT8 ...")
ckpt8 = f"{WORKING_DIR}/models/checkpoints/standard_qat_int8/final.pt"
m8 = load_qat_for_inference(qat8_cfg, ckpt8)
int8_outs = generate_with_model(m8)
del m8; free()

# 3. Standard QAT INT4
print("Generating Standard QAT INT4 ...")
ckpt4 = f"{WORKING_DIR}/models/checkpoints/standard_qat_int4/final.pt"
m4 = load_qat_for_inference(qat4_cfg, ckpt4)
int4_outs = generate_with_model(m4)
del m4; free()

print("Done.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

inference_df = pd.DataFrame({
    "Prompt":              SAMPLE_PROMPTS,
    "FP32":                fp32_outs,
    "Standard QAT INT8":   int8_outs,
    "Standard QAT INT4":   int4_outs,
})

inference_df.to_json(Path(REPO_DIR) / "results" / "standard_qat_inference.json",
                     orient="records", indent=2)

display(HTML(inference_df.to_html(index=False, escape=False).replace(
    "<table", '<table style="font-family:monospace; font-size:12px"')))

## Export everything to Drive

Bundles five things into `/content/drive/MyDrive/sqat-standard-qat/`:

1. **Checkpoints** — `models/checkpoints/standard_qat_int{4,8}.pt` (renamed from `final.pt` for portability)
2. **Datasets** — small samples (8 chunks each) of the actual tokenized train/val/test data the trainer used
3. **Metric summary** — `results/metric_summary.csv` (one row per variant: PPL, KLD, train time, final loss)
4. **Per-step loss** (micro view) — `results/logs/qat_int{4,8}/per_step_loss.jsonl`
5. **Eval-interval log** (macro view) — `results/logs/qat_int{4,8}/training_steps.jsonl`

Notebook 06 (export) reads checkpoints from this Drive folder; notebook 07 reads the JSON/CSV summaries.

In [ ]:
import shutil
import pandas as pd

dst = Path(DRIVE_ROOT)
(dst / "models" / "checkpoints").mkdir(parents=True, exist_ok=True)
(dst / "data_samples").mkdir(parents=True, exist_ok=True)

# 1. Rename + copy checkpoints
for bits, label in [(8, "int8"), (4, "int4")]:
    src = Path(REPO_DIR) / "models" / "checkpoints" / f"standard_qat_{label}" / "final.pt"
    out = dst / "models" / "checkpoints" / f"standard_qat_{label}.pt"
    if src.exists():
        shutil.copy2(src, out)
        size_mb = out.stat().st_size / 1e6
        print(f"  {out}  ({size_mb:.0f} MB)")
    else:
        print(f"  SKIP — checkpoint not found: {src}")

# 2. Save dataset samples (the actual tokenized chunks the trainer saw)
from src.utils.config_loader import load_config
from src.utils.data_loader import build_dataloaders, build_validation_loader

cfg = load_config(str(qat4_cfg))
train_loader, eval_loader = build_dataloaders(cfg, num_workers=0)
val_loader = build_validation_loader(cfg, num_workers=0)

def sample_loader(loader, n=8):
    chunks = []
    for batch in loader:
        chunks.append({"input_ids": batch["input_ids"][:n].cpu().clone()})
        if sum(c["input_ids"].size(0) for c in chunks) >= n:
            break
    return torch.cat([c["input_ids"] for c in chunks], dim=0)[:n]

torch.save({"input_ids": sample_loader(train_loader), "split": "train", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "train_sample.pt")
torch.save({"input_ids": sample_loader(val_loader),   "split": "validation", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "val_sample.pt")
torch.save({"input_ids": sample_loader(eval_loader),  "split": "test", "seq_length": cfg.data.seq_length},
           dst / "data_samples" / "test_sample.pt")
print(f"  Dataset samples written to {dst}/data_samples/")

# 3. Metric summary CSV (compact, one row per variant)
metric_rows = []
for label, r in [("standard_qat_int8", results_int8), ("standard_qat_int4", results_int4)]:
    metric_rows.append({
        "experiment":         label,
        "perplexity":         r.get("perplexity"),
        "kl_divergence":      r.get("kl_divergence"),
        "final_loss":         r.get("final_loss"),
        "training_time_s":    r.get("training_time_seconds"),
        "total_steps":        r.get("total_steps"),
    })
metric_df = pd.DataFrame(metric_rows).round(6)
metric_csv = dst / "results" / "metric_summary.csv"
metric_csv.parent.mkdir(parents=True, exist_ok=True)
metric_df.to_csv(metric_csv, index=False)
print(f"  Metric summary -> {metric_csv}")
display(metric_df)

# 4-5. Copy results dir (includes per_step_loss.jsonl + training_steps.jsonl + JSONs)
shutil.copytree(f"{REPO_DIR}/results", dst / "results", dirs_exist_ok=True)
shutil.copytree(COLAB_CFG, dst / "configs_colab" / "standard_qat", dirs_exist_ok=True)

print(f"\nAll outputs saved to: {dst}")
!du -sh {dst}